# RCA Curved-Reformat Metadata Diagnostic — clean main baseline
This notebook asks whether Siemens series **1035 (RCA Curved Range Radial Q3D)** contains enough standard/private DICOM metadata to recover its source series and/or vessel trajectory in the true axial CCTA coordinate system. It does **not** implement a new centerline algorithm.

Use **Runtime → Run all**. Google Drive mounts first.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Clone clean branch and install dependencies before third-party imports.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom

import os, time, shutil, zipfile, re
from pathlib import Path
from collections import defaultdict, Counter
import pydicom
from pydicom.dataset import Dataset
print('Dependencies ready.')


## Stage and scan the study locally
The ZIP is copied from Drive to local Colab disk before extraction and metadata scanning.

In [ ]:
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT=Path('/content/full_dicom_metadata_diag')
if not DRIVE_ZIP.exists(): raise FileNotFoundError(f'Missing {DRIVE_ZIP}')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying {DRIVE_ZIP.name} to local disk...',flush=True)
    t=time.time(); shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP); print(f'Copy finished in {time.time()-t:.1f}s',flush=True)
else: print('Local ZIP already staged.')
if EXTRACT.exists(): shutil.rmtree(EXTRACT)
print('Extracting locally...',flush=True)
t=time.time()
with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(EXTRACT)
print(f'Extraction finished in {time.time()-t:.1f}s',flush=True)

groups=defaultdict(list); first_headers={}
print('Scanning DICOM headers...',flush=True)
t=time.time()
for root,_,files in os.walk(EXTRACT):
    for fn in files:
        p=os.path.join(root,fn)
        try:
            ds=pydicom.dcmread(p,stop_before_pixels=True,force=True)
            uid=str(ds.SeriesInstanceUID)
            groups[uid].append(p)
            first_headers.setdefault(uid,ds)
        except Exception: pass
print(f'Header scan finished in {time.time()-t:.1f}s; {len(groups)} series found.')

def uid_for_series(number):
    hits=[uid for uid,ds in first_headers.items() if int(getattr(ds,'SeriesNumber',-1))==number]
    if not hits: raise ValueError(f'Series {number} not found')
    return hits[0]
CURVED_UID=uid_for_series(1035)
SOURCE_UID=uid_for_series(7)
print('RCA curved series UID:',CURVED_UID)
print('Source CCTA series 7 UID:',SOURCE_UID)
print('RCA curved images:',len(groups[CURVED_UID]),' source images:',len(groups[SOURCE_UID]))


## Standard DICOM references
This prints standard geometry/derivation/reference fields from several RCA Q3D frames and searches recursively for references to source series 7.

In [ ]:
def short(v,n=240):
    s=str(v).replace('\n',' ')
    return s if len(s)<=n else s[:n]+' ...'

def walk(ds,prefix='',depth=0,max_depth=5):
    if depth>max_depth: return
    for elem in ds:
        path=f'{prefix}/{elem.tag} {elem.keyword or elem.name}'
        if elem.VR=='SQ':
            yield path,'SQ',f'{len(elem.value)} item(s)'
            for i,item in enumerate(elem.value):
                yield from walk(item,f'{path}[{i}]',depth+1,max_depth)
        else:
            yield path,elem.VR,elem.value

files1035=groups[CURVED_UID]
sample_indices=sorted(set([0,len(files1035)//2,len(files1035)-1]))
standard_names=['SOPClassUID','SOPInstanceUID','SeriesInstanceUID','SeriesNumber','SeriesDescription','ImageType','DerivationDescription','FrameOfReferenceUID','ImageOrientationPatient','ImagePositionPatient','PixelSpacing','SliceThickness','SpacingBetweenSlices','SpatialLocationsPreserved']
for i in sample_indices:
    ds=pydicom.dcmread(files1035[i],stop_before_pixels=True,force=True)
    print(f'\n===== RCA 1035 frame {i} =====')
    for name in standard_names:
        print(f'{name}:',short(getattr(ds,name,None)))
    ref_hits=[]
    for path,vr,val in walk(ds):
        s=str(val)
        if SOURCE_UID in s or any(k in path.lower() for k in ['source image','referenced series','derivation image','referenced image']):
            ref_hits.append((path,vr,short(val)))
    print('Reference-related elements:')
    for hit in ref_hits[:80]: print(' ',hit)
    if not ref_hits: print('  none found')


## Private metadata search
Search Siemens/private tags for words or compact numeric fields suggestive of curved-path coordinates, centerlines, Q3D geometry, or source references. Large opaque byte blocks are summarized rather than dumped.

In [ ]:
KEYWORDS=('center','centre','curve','curved','path','trajectory','vessel','coronary','rca','q3d','range','source','reference','position','coordinate','geometry','syngo')
def private_summary(ds):
    out=[]
    creators=set()
    for path,vr,val in walk(ds,max_depth=8):
        # Recover element from textual path only for filtering; path includes private tag/name.
        pl=path.lower(); sv=short(val,320); sl=sv.lower()
        is_interesting=any(k in pl or k in sl for k in KEYWORDS) or SOURCE_UID in str(val)
        # Private tags are recognizable by odd group numbers in '(gggg,eeee)'.
        m=re.findall(r'\(([0-9a-fA-F]{4}),([0-9a-fA-F]{4})\)',path)
        is_private=any(int(g,16)%2==1 for g,e in m)
        if is_private and is_interesting:
            out.append((path,vr,sv))
    # Also list private creator strings directly from the dataset.
    for elem in ds.iterall():
        if elem.tag.is_private and elem.VR not in ('OB','OW','OF','OD','UN','SQ'):
            txt=short(elem.value,160)
            if elem.tag.element>=0x0010 and elem.tag.element<=0x00FF:
                creators.add(txt)
    return sorted(creators),out

all_hits=[]; all_creators=Counter()
for idx,p in enumerate(files1035):
    ds=pydicom.dcmread(p,stop_before_pixels=True,force=True)
    creators,hits=private_summary(ds)
    all_creators.update(creators)
    for h in hits: all_hits.append((idx,)+h)
print('Private creators seen in RCA 1035:')
for c,n in all_creators.most_common(): print(f' {n:>3}  {c}')
print('\nInteresting private/reference metadata hits (first 160):')
for h in all_hits[:160]: print(h)
print('Total interesting hits:',len(all_hits))


## Direct UID linkage test
This checks whether any RCA Q3D metadata contains the source **SeriesInstanceUID** or any **SOPInstanceUID** from the selected best-diastolic source series 7.

In [ ]:
source_sops=set()
for p in groups[SOURCE_UID]:
    try:
        ds=pydicom.dcmread(p,stop_before_pixels=True,force=True,specific_tags=['SOPInstanceUID'])
        if hasattr(ds,'SOPInstanceUID'): source_sops.add(str(ds.SOPInstanceUID))
    except Exception: pass
series_uid_hits=[]; sop_hits=[]
for idx,p in enumerate(files1035):
    ds=pydicom.dcmread(p,stop_before_pixels=True,force=True)
    for path,vr,val in walk(ds,max_depth=10):
        sval=str(val)
        if SOURCE_UID in sval: series_uid_hits.append((idx,path,short(val)))
        for sop in source_sops:
            if sop in sval:
                sop_hits.append((idx,path,sop)); break
print('Source SeriesInstanceUID hits:',len(series_uid_hits))
for h in series_uid_hits[:40]: print(' ',h)
print('Source SOPInstanceUID hits:',len(sop_hits))
for h in sop_hits[:40]: print(' ',h)


## Save a compact report to Drive
The full notebook output remains visible in Colab; this text file records the key linkage result and private creators.

In [ ]:
REPORT=ROOT/'RCA_Curved_Metadata_Diagnostic.txt'
with open(REPORT,'w') as f:
    f.write('OpenPlaque RCA curved-reformat metadata diagnostic\n')
    f.write('Clean branch derived directly from main\n\n')
    f.write(f'Curved series 1035 UID: {CURVED_UID}\n')
    f.write(f'Source series 7 UID: {SOURCE_UID}\n')
    f.write(f'Source SeriesInstanceUID hits in 1035 metadata: {len(series_uid_hits)}\n')
    f.write(f'Source SOPInstanceUID hits in 1035 metadata: {len(sop_hits)}\n')
    f.write('\nPrivate creators:\n')
    for c,n in all_creators.most_common(): f.write(f'{n:3d} {c}\n')
    f.write('\nInteresting hits (first 160):\n')
    for h in all_hits[:160]: f.write(repr(h)+'\n')
print('Saved:',REPORT)
print('METADATA DIAGNOSTIC COMPLETE.')
print('Please send the Private creators / Interesting hits / UID linkage output.')
